In [1]:
!pip install yfinance scikit-learn pandas numpy -q

In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import pickle
import warnings
warnings.filterwarnings('ignore')

In [3]:
tickers    = ["BBCA.JK", "BBRI.JK", "BMRI.JK"]
start_date = "2019-11-20"
end_date   = "2026-01-20"

In [4]:
raw = yf.download(tickers, start=start_date, end=end_date, progress=False)

price_data = {}
for ticker in tickers:
    df = raw.loc[:, (slice(None), ticker)].copy()
    df.columns = df.columns.droplevel(1)
    df = df[['Open', 'High', 'Low', 'Close', 'Volume']]
    df.index = pd.to_datetime(df.index)
    price_data[ticker] = df


1 Failed download:
['BMRI.JK']: OperationalError('database is locked')


In [5]:
price_data['BBCA.JK'].tail()

Price,Open,High,Low,Close,Volume
Date,,,,,
2026-01-12,7736.332185,7760.283678,7664.477706,7688.429199,82935700
2026-01-13,7688.429532,7736.332520,7664.478038,7736.332520,80536300
2026-01-14,7688.429521,7712.381015,7616.575040,7664.478027,105724300
2026-01-15,7664.478038,7832.138495,7640.526544,7736.332520,151753100
2026-01-19,7736.332365,7784.235352,7640.526391,7784.235352,172100000


In [6]:
fundamental = {}

for ticker in tickers:
    info = yf.Ticker(ticker).info
    fundamental[ticker] = {
        'ROE': info.get('returnOnEquity'),
        'EPS': info.get('trailingEps'),
        'DY' : info.get('dividendYield') / 100
    }

In [7]:
fundamental

{'BBCA.JK': {'ROE': 0.22972, 'EPS': 466.83, 'DY': 0.054400000000000004},
 'BBRI.JK': {'ROE': 0.18135999, 'EPS': 388.92, 'DY': 0.1282},
 'BMRI.JK': {'ROE': 0.21040002, 'EPS': 603.29, 'DY': 0.10300000000000001}}

In [8]:
merged_data = {}

for ticker in tickers:
    df = price_data[ticker].copy()
    df['ROE'] = fundamental[ticker]['ROE']
    df['EPS'] = fundamental[ticker]['EPS']
    df['DY']  = fundamental[ticker]['DY']
    merged_data[ticker] = df

In [9]:
merged_data['BBCA.JK'].tail()

Price,Open,High,Low,Close,Volume,ROE,EPS,DY
Date,,,,,,,,
2026-01-12,7736.332185,7760.283678,7664.477706,7688.429199,82935700,0.22972,466.83,0.0544
2026-01-13,7688.429532,7736.332520,7664.478038,7736.332520,80536300,0.22972,466.83,0.0544
2026-01-14,7688.429521,7712.381015,7616.575040,7664.478027,105724300,0.22972,466.83,0.0544
2026-01-15,7664.478038,7832.138495,7640.526544,7736.332520,151753100,0.22972,466.83,0.0544
2026-01-19,7736.332365,7784.235352,7640.526391,7784.235352,172100000,0.22972,466.83,0.0544


In [10]:
def add_features(df):
    df = df.copy()
    
    df['MA7']  = df['Close'].rolling(7).mean()
    df['MA20'] = df['Close'].rolling(20).mean()
    df['MA50'] = df['Close'].rolling(50).mean()

    delta = df['Close'].diff()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    df['RSI'] = 100 - (100 / (1 + gain / (loss + 1e-9)))

    sma20          = df['Close'].rolling(20).mean()
    std20          = df['Close'].rolling(20).std()
    df['BB_upper'] = sma20 + 2 * std20
    df['BB_lower'] = sma20 - 2 * std20
    df['BB_width'] = df['BB_upper'] - df['BB_lower']

    df['Daily_Return'] = df['Close'].pct_change()
    df['Log_Return']   = np.log(df['Close'] / df['Close'].shift(1))
    df['Volume_MA7']   = df['Volume'].rolling(7).mean()

    ema12          = df['Close'].ewm(span=12, adjust=False).mean()
    ema26          = df['Close'].ewm(span=26, adjust=False).mean()
    df['MACD']         = ema12 - ema26
    df['MACD_signal']  = df['MACD'].ewm(span=9, adjust=False).mean()

    df.dropna(inplace=True)
    return df

featured_data = {}
for ticker in tickers:
    featured_data[ticker] = add_features(merged_data[ticker])

In [11]:
featured_data['BBCA.JK'].columns.tolist()

['Open',
 'High',
 'Low',
 'Close',
 'Volume',
 'ROE',
 'EPS',
 'DY',
 'MA7',
 'MA20',
 'MA50',
 'RSI',
 'BB_upper',
 'BB_lower',
 'BB_width',
 'Daily_Return',
 'Log_Return',
 'Volume_MA7',
 'MACD',
 'MACD_signal']

In [12]:
feature_cols = [
    'Open','High','Low','Close','Volume',
    'ROE','EPS','DY',
    'MA7','MA20','MA50',
    'RSI','BB_upper','BB_lower','BB_width',
    'Daily_Return','Log_Return','Volume_MA7',
    'MACD','MACD_signal'
]

scaled_data = {}
scalers     = {}

for ticker in tickers:
    df     = featured_data[ticker][feature_cols].copy()
    scaler = MinMaxScaler(feature_range=(0, 1))
    arr    = scaler.fit_transform(df)
    scaled_data[ticker] = pd.DataFrame(arr, columns=feature_cols, index=df.index)
    scalers[ticker]     = scaler

ValueError: Found array with 0 sample(s) (shape=(0, 20)) while a minimum of 1 is required by MinMaxScaler.

In [ ]:
scaled_data['BBCA.JK'].tail()

In [ ]:
WINDOW_SIZE = 60
TARGET_COL  = 'Close'
TARGET_IDX  = feature_cols.index(TARGET_COL)

def create_sequences(scaled_df, window_size, target_idx):
    X, y = [], []
    arr  = scaled_df.values
    for i in range(window_size, len(arr)):
        X.append(arr[i - window_size : i, :])
        y.append(arr[i, target_idx])
    return np.array(X), np.array(y)

sequences = {}

for ticker in tickers:
    X, y  = create_sequences(scaled_data[ticker], WINDOW_SIZE, TARGET_IDX)
    split = int(len(X) * 0.8)
    sequences[ticker] = {
        'X_train': X[:split],  'y_train': y[:split],
        'X_test' : X[split:],  'y_test' : y[split:],
    }

In [ ]:
for ticker in tickers:
    s = sequences[ticker]
    print(f"{ticker} → X_train: {s['X_train'].shape}, X_test: {s['X_test'].shape}")

In [ ]:
output = {
    'sequences'           : sequences,
    'scalers'             : scalers,
    'feature_cols'        : feature_cols,
    'target_col'          : TARGET_COL,
    'target_idx'          : TARGET_IDX,
    'window_size'         : WINDOW_SIZE,
    'featured_data'       : featured_data,
    'fundamental'         : fundamental,
    'tickers'             : tickers,
}

with open('preprocessed_data.pkl', 'wb') as f:
    pickle.dump(output, f)

In [ ]:
import shutil
shutil.copy('preprocessed_data.pkl', '/kaggle/working/preprocessed_data.pkl')

for ticker in tickers:
    featured_data[ticker].to_csv(f'/kaggle/working/{ticker.replace(".","_")}_featured.csv')